# MIP Python Client — Getting Started

Welcome to the MIP Jupyter workspace. The `mip` client is pre-installed and connects to platform-backend under `/services`.

**Run all cells from top to bottom** to verify your connection and run a simple analysis.

| Where to go next | Path |
|------------------|------|
| Example analysis | `examples/feres_analysis.ipynb` |
| Documentation | `docs/` (quickstart, API, troubleshooting) |
| Your notebooks | `scratch/` |

Reference: `docs/quickstart.md`

## Prerequisites

**Platform UI:** `PLATFORM_BACKEND_URL` and `MIP_TOKEN` are injected when you open notebooks from the portal.

**Local development:** Set `PLATFORM_BACKEND_URL` (or `MIP_BASE_URL`) and `MIP_TOKEN` (or `PLATFORM_TOKEN`) before starting Jupyter.

If `Client.from_env()` fails, see `docs/troubleshooting.md`.

In [1]:
import mip
from mip.filters import F
from mip.preprocessing import MissingValuesHandler, OutlierWinsorizer

client = mip.Client.from_env()
catalog = client.catalog()
print("Public exports:", ", ".join(mip.__all__))

Public exports: AnalysisSet, Client, MipAlgorithmNotAvailable, MipBackendError, MipConfigurationError, MipError, Pipeline, UnsupportedOperationError


## Catalog discovery

List all authorized data models, inspect the catalog tree, then pick one to explore datasets and variables.

In [2]:
catalog.list()
catalog.summaries()
catalog.tree()

Data models (3)
|-- Dementia:0.1 [33 datasets, 2 root vars, 26 groups, 182 grouped vars]
|-- Dementia longitudinal:0.1 [4 datasets, 4 root vars, 26 groups, 181 grouped vars]
`-- Traumatic Brain Injury:0.1 [11 datasets, 1 root vars, 6 groups, 20 grouped vars]

In [11]:
dm = catalog.data_model("dementia")
dm.summary()
dm.list_datasets()

dm.datasets.list()
dm.list_groups()

dm.tree()
dm.tree(group="demographics", include_variables=True)


Metadata tree for dementia:0.1 (Dementia) > demographics
`-- Demographics [5 vars]
   |-- agegroup [nominal]
   |-- Gender [nominal]
   |-- Handedness [nominal]
   |-- Exact age [real]
   `-- Age Years [integer]

## Analysis and pipeline

In [ ]:
adni = dm.datasets["adni"]
age = dm.variables["age"]
mmse = dm.variables["mmse"]
diagnosis = dm.variables["diagnosis"]

analysis_set = mip.AnalysisSet(
    data_model=dm,
    datasets=[adni],
    variables=[age, mmse, diagnosis],
)
analysis_set.summary()

In [ ]:
pipeline = mip.Pipeline(
    analysis_set=analysis_set,
    filters=(F(age) >= 50) & F(mmse).is_not_null(),
    handle_missing=MissingValuesHandler(strategies={mmse: "mean"}),
    outlier_handling=OutlierWinsorizer(strategies={mmse: "iqr"}),
)
pipeline.explain()

In [ ]:
histogram = pipeline.histogram(variable=mmse, bins=20)
histogram.summary()

## Algorithm registry and sklearn export


In [ ]:
client.algorithms().search("logistic")
# Logistic regression results can export to sklearn when backend output includes feature names and coefficients.
# sklearn_model = logreg_result.to_sklearn()


## Next steps

- **Stroke analysis example:** `examples/feres_analysis.ipynb`
- **API reference:** `docs/api-reference.md`
- **Your own work:** copy a template into `scratch/` and adapt it